In [1]:
import mypy
from typing import Dict, List

# Problemen/Problems

De chunking functies werken nog niet echt goed. Voor WerkwijzerPoortwachter heb ik een probleem met het herkennen van hoofdstukken, omdat de opsommingen hetzelfde formaat gebruiken.

# Document Preprocessing

In this file, we want to pre-process the documents which will be fed to the RAG through a vector database.
We have different file formats, and different types of sources. 
Each heading will contain information about the data source and the filetype.
But before we start with actually processing the text, let's first read each file into a string.

## Reading the data
For now the following data types are accepted:

- .txt
- .pdf

Let us store all the text strings in a dictionary for now.

In [2]:
source_strings = {}

Now write a function which allows to read in a string from an information source (assume each information source has a unique name)

In [3]:
from PyPDF2 import PdfReader
from pathlib import Path

#For .txt files:
def read_txt(txt_file, source_strings):
    with open(txt_file, "r", encoding="utf-8") as f:
        source_strings[str(txt_file)] = f.read()

def read_pdf(pdf_file, source_strings):
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    source_strings[str(pdf_file)] = text

def read_file(file, source_strings):
    if file.suffix == '.pdf':
        read_pdf(file, source_strings)
    elif file.suffix == '.txt':
        read_txt(file, source_strings)

Let us read in all the data we can find under Documents

In [4]:
#Main directory, containing a folder for each information source
directory = r"Documents/"

folders = Path(directory).glob("*")
for folder in folders:
    for file in folder.glob("*"):
        read_file(file, source_strings)

In [5]:
print(source_strings.keys())

dict_keys(['Documents\\WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf', 'Documents\\Wet Verbetering Poortwachter\\Arbeidsomstandighedenwet.txt', 'Documents\\Wet Verbetering Poortwachter\\BurgerlijkWetboekBoek7.txt', 'Documents\\Wet Verbetering Poortwachter\\Werkloosheidswet.txt', 'Documents\\Wet Verbetering Poortwachter\\WetArbeidsongeschiktheidsverzekeringZelfstandigen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetArbeidsongeschiktheidsvoorzieningJongghandicapten.txt', 'Documents\\Wet Verbetering Poortwachter\\WetBeslistermijnenSocialeVerzekeringen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetOpDeArbeidsongeschiktheidsverzekering.txt', 'Documents\\Wet Verbetering Poortwachter\\WetOverheidspersoneelOnderDeWerkenemersverzekeringen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetTerugdringingZiekteverzuim.txt', 'Documents\\Wet Verbetering Poortwachter\\Ziektewet.txt'])


Now `source_strings` contains a dictionary of filename:text for all the files in our collection.

## Cleaning the data

Some of our data contains leading white space and or useless text.

In [6]:
print(next(iter(source_strings.values()))[0:70])

Werkwijzer Poortwachter
UWV, 1 augustus 2022 
Bij de geactualiseerde W


Let's remove this. And let's remove the useless characters [ and ]

In [7]:
for key in source_strings.keys():
    #Strip leading and ending whitespace and newlines
    source_strings[key] = source_strings[key].lstrip('\r\n ')
    #The special characters [ and ] do not add extra context.
    #Also convert everything to lower case.
    source_strings[key] = source_strings[key].replace("[", "").replace("]", "").lower()

Let's remove the table of contents from WerkwijzerPoortwachter

In [8]:
source_strings['Documents\\WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf'] = source_strings['Documents\\WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf'][source_strings['Documents\\WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf'].find('1. inleiding\n'):]


## Chunking the data

Having extracted the data, we now want to chunk it. For this, we probably take a different approach for each information source

We write different chunking functions, depending on the information source. Then we map them to each other.

In [9]:
import re

#Gebruikt voor wet verbetering poortwachter
def chunk_text_with_metadata(text: str, filename: str) -> list:
    # Regex patterns for structure recognition
    hoofdstuk_pattern = re.compile(r'^Hoofdstuk\s+\d+[:.]?\s*(.*)$', re.IGNORECASE)
    artikel_pattern = re.compile(r'^Artikel\s+\d+[\.\d]*[:.]?\s*(.*)$', re.IGNORECASE)
    sub_paragraaf_pattern = re.compile(r'^(\d+[\.\d]*)\s+(.*)$')
    chunks = []
    current_hoofdstuk = None
    current_artikel = None
    lines = text.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Check for hoofdstuk
        hoofdstuk_match = hoofdstuk_pattern.match(line)
        if hoofdstuk_match:
            current_hoofdstuk = hoofdstuk_match.group(0)
            current_artikel = None
        # Check for artikel
        artikel_match = artikel_pattern.match(line)
        if artikel_match:
            current_artikel = artikel_match.group(0)
        # Check for sub-paragraph
        sub_paragraaf_match = sub_paragraaf_pattern.match(line)
        if sub_paragraaf_match:
            sub_nummer = sub_paragraaf_match.group(1)
            sub_tekst = sub_paragraaf_match.group(2)
            chunks.append({
                "filename": filename,
                "hoofdstuk": current_hoofdstuk,
                "artikel": current_artikel,
                "sub_paragraaf": sub_nummer,
                "text": sub_tekst
            })
        else:
            # No sub-paragraph match, chunk the whole line
            chunks.append({
                "filename": filename,
                "hoofdstuk": current_hoofdstuk,
                "artikel": current_artikel,
                "sub_paragraaf": None,
                "text": line
            })
    return chunks




import re

def chunk_text_with_consistent_hoofdstukken(text: str, filename: str) -> list:
    chunks = []
    current_hoofdstuk = None
    last_hoofdstuk_nummer = 0  # Houdt het laatste hoofdstuknummer bij
    last_hoofdstuk_line = 0

    lines = text.split('\n')
    for idx, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue  # Sla lege regels over

        # Controleer of de regel een hoofdstuk is
        hoofdstuk_match = re.match(r'^(\d+)\.\s+(.*)$', line)
        if hoofdstuk_match:
            hoofdstuk_nummer = int(hoofdstuk_match.group(1))
            hoofdstuk_naam = hoofdstuk_match.group(2)

            # Controleer of het hoofdstuknummer consistent omhoog gaat
            if hoofdstuk_nummer <= last_hoofdstuk_nummer:
                continue

            if hoofdstuk_nummer != last_hoofdstuk_nummer +1:
                continue

            if idx == last_hoofdstuk_line + 1:
                continue

            current_hoofdstuk = f"{hoofdstuk_nummer}. {hoofdstuk_naam}"
            last_hoofdstuk_line = idx
            last_hoofdstuk_nummer = hoofdstuk_nummer
            continue

        # Voeg de tekstregel toe als chunk met de huidige hoofdstukmetadata
        chunks.append({
            "filename": filename,
            "hoofdstuk": current_hoofdstuk,
            "text": line
        })

    return chunks




In [10]:
FOLDER_TO_CHUNKER = {
    "Wet Verbetering Poortwachter": chunk_text_with_metadata,  
    "WerkwijzerPoortwachter": chunk_text_with_consistent_hoofdstukken,
    # Voeg hier nieuwe foldernamen en functies toe
}

In [11]:
import os

def process_all_files(source_strings: dict) -> list:
    all_chunks = []
    for file_path, text in source_strings.items():
        # Extraheer de foldernaam uit het pad
        folder_name = os.path.basename(os.path.dirname(file_path))

        # Haal de bijbehorende chunking-functie op uit de mapping
        chunker = FOLDER_TO_CHUNKER.get(folder_name)
        if chunker is None:
            raise ValueError(f"Geen chunking-functie gedefinieerd voor folder: {folder_name}")

        # Roep de juiste chunking-functie aan
        chunks = chunker(text, file_path)
        all_chunks.extend(chunks)

    return all_chunks


In [12]:
import json

def save_chunks_to_json(chunks: list, output_file: str):
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)


In [13]:
# Process all files
all_chunks = process_all_files(source_strings)

# Save to JSON
save_chunks_to_json(all_chunks, "structured_chunks.json")

Lijkt best goed te werken! Laten we het hier voor nu bij houden.

In [14]:
print(source_strings.keys())

dict_keys(['Documents\\WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf', 'Documents\\Wet Verbetering Poortwachter\\Arbeidsomstandighedenwet.txt', 'Documents\\Wet Verbetering Poortwachter\\BurgerlijkWetboekBoek7.txt', 'Documents\\Wet Verbetering Poortwachter\\Werkloosheidswet.txt', 'Documents\\Wet Verbetering Poortwachter\\WetArbeidsongeschiktheidsverzekeringZelfstandigen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetArbeidsongeschiktheidsvoorzieningJongghandicapten.txt', 'Documents\\Wet Verbetering Poortwachter\\WetBeslistermijnenSocialeVerzekeringen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetOpDeArbeidsongeschiktheidsverzekering.txt', 'Documents\\Wet Verbetering Poortwachter\\WetOverheidspersoneelOnderDeWerkenemersverzekeringen.txt', 'Documents\\Wet Verbetering Poortwachter\\WetTerugdringingZiekteverzuim.txt', 'Documents\\Wet Verbetering Poortwachter\\Ziektewet.txt'])
